In [8]:
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib  # Added for generating the content hash
from datetime import datetime  # Added for full timestamps

BASE_URL = "https://www.kanker.nl"
START_URL = "https://www.kanker.nl/kankersoorten"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
MAX_WORKERS = 5  # Controls how many pages to scrape at the same time

def get_cancer_links():
    print(f"Fetching overview page: {START_URL}")
    try:
        response = requests.get(START_URL, headers=HEADERS, timeout=10)
        if response.status_code != 200:
            return {}
        soup = BeautifulSoup(response.text, 'lxml') # 'lxml' is faster than 'html.parser'
        cancer_links = {}
        grid_container = soup.find('div', class_='dynamic-grid') or soup.find('main') or soup

        for link in grid_container.find_all('a', href=True):
            href = link['href']
            if href.startswith('/kankersoorten/') and len(href.split('/')) == 3:
                name = link.get_text(strip=True)
                if name and not any(s in name.lower() for s in ["bekijk alle", "meer kankersoorten"]):
                    if name not in cancer_links:
                        cancer_links[name] = BASE_URL + href
        return cancer_links
    except Exception as e:
        print(f"Error fetching directory: {e}")
        return {}

def scrape_comprehensive_cancer(cancer_name, hub_url):
    print(f"\n[Processing Category] {cancer_name} -> {hub_url}")
    
    # Generate timestamp down to the second right as processing begins
    scrape_date = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    data_row = {
        "Cancer Type": cancer_name,
        "Source URL": hub_url,
        "Scrape Date": scrape_date,
        "Content Hash": "N/A",
        "General Description": "N/A",
        "Prognosis (Vooruitzichten)": "N/A",
        "Symptoms (Symptomen)": "N/A",
        "Causes (Oorzaken)": "N/A",
        "Treatments (Behandeling)": "N/A"
    }

    try:
        response = requests.get(hub_url, headers=HEADERS, timeout=10)
        if response.status_code != 200:
            return data_row
            
        soup = BeautifulSoup(response.text, 'lxml')
        
        # --- FIX 1: DYNAMIC SUB-LINK HARVESTING ---
        target_urls = [hub_url]
        base_path = hub_url.replace(BASE_URL, '') # e.g., "/kankersoorten/buikvlieskanker"
        
        # Find every anchor tag inside the main layout
        for a in soup.find_all('a', href=True):
            sub_href = a['href']
            # Allow links deeper in the directory chain (removed length restriction)
            if sub_href.startswith(base_path):
                full_sub_url = BASE_URL + sub_href if sub_href.startswith('/') else sub_href
                if full_sub_url not in target_urls and "#" not in sub_href:
                    target_urls.append(full_sub_url)

        print(f"--> Discovered {len(target_urls)} pages to scan for content.")

        # Crawl each discovered child tracking layout page
        for current_url in target_urls:
            sub_res = requests.get(current_url, headers=HEADERS, timeout=10)
            if sub_res.status_code != 200:
                continue
                
            sub_soup = BeautifulSoup(sub_res.text, 'lxml')
            main_article = sub_soup.find('article') or sub_soup.find('main') or sub_soup
            
            # --- INTRODUCTION CAPTURE ---
            if data_row["General Description"] == "N/A":
                for p in main_article.find_all('p'):
                    p_txt = p.get_text(strip=True)
                    if not p_txt or "lees op deze pagina" in p_txt.lower():
                        continue
                    if not any(phrase in p_txt.lower() for phrase in ["deze informatie is gecontroleerd", "printen", "opslaan"]):
                        data_row["General Description"] = p_txt
                        break 

            # --- FIX 2: BULLETPROOF HEADING TEXT ACQUISITION ---
            # Extract all child block components flat inside the main canvas area
            elements = main_article.find_all(['h1', 'h2', 'h3', 'p', 'ul', 'ol'])
            
            current_heading_category = None
            section_accumulator = []

            for element in elements:
                if element.name in ['h1', 'h2', 'h3']:
                    # First, save data collected from the *previous* category block before switching
                    if current_heading_category and section_accumulator:
                        merged_txt = " ".join(section_accumulator)
                        if data_row[current_heading_category] == "N/A" and len(merged_txt) > 15:
                            data_row[current_heading_category] = merged_txt
                    
                    # Reset tracker states for the new header text match
                    heading_text = element.get_text(strip=True).lower()
                    current_heading_category = None
                    section_accumulator = []

                    # Determine mapping target key strings
                    if any(k in heading_text for k in ['vooruitzichten', 'prognose', 'beter worden']):
                        current_heading_category = "Prognosis (Vooruitzichten)"
                    elif any(k in heading_text for k in ['symptomen', 'klachten', 'hoe ziet', 'eruit']):
                        current_heading_category = "Symptoms (Symptomen)"
                    elif any(k in heading_text for k in ['oorzaken', 'risicofactoren', 'ontstaan']):
                        current_heading_category = "Causes (Oorzaken)"
                    elif any(k in heading_text for k in ['behandeling', 'operatie', 'bestraling', 'chemo', 'hipec']):
                        current_heading_category = "Treatments (Behandeling)"
                        
                elif current_heading_category:
                    # If tracking a matching category header, append all structural paragraphs/lists found below it
                    txt = element.get_text(" ", strip=True)
                    if txt and not any(phrase in txt.lower() for phrase in ["lees verder over", "vraag het een professional"]):
                        if txt not in section_accumulator:
                            section_accumulator.append(txt)

            # Final check cleanup loop termination step save
            if current_heading_category and section_accumulator:
                merged_txt = " ".join(section_accumulator)
                if data_row[current_heading_category] == "N/A" and len(merged_txt) > 15:
                    data_row[current_heading_category] = merged_txt

        # --- CONTENT HASH GENERATION ---
        # Concatenate content text blocks to generate a reliable row signature string
        combined_text = (
            f"{data_row['General Description']}\n"
            f"{data_row['Prognosis (Vooruitzichten)']}\n"
            f"{data_row['Symptoms (Symptomen)']}\n"
            f"{data_row['Causes (Oorzaken)']}\n"
            f"{data_row['Treatments (Behandeling)']}"
        )
        data_row["Content Hash"] = hashlib.sha256(combined_text.encode('utf-8')).hexdigest()

    except Exception as e:
        print(f"Error extracting profile contents for {cancer_name}: {e}")

    return data_row

def main():
    start_time = time.time()
    cancer_directory = get_cancer_links()
    if not cancer_directory:
        print("Directory index missing.")
        return

    all_cancer_data = []
    
    # Run requests in parallel using ThreadPoolExecutor
    print(f"\nStarting parallel scraping with {MAX_WORKERS} workers...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_url = {
            executor.submit(scrape_comprehensive_cancer, name, url): name 
            for name, url in cancer_directory.items()
        }
        
        for future in as_completed(future_to_url):
            data_row = future.result()
            all_cancer_data.append(data_row)

    # Export to files with strict column ordering constraints
    df = pd.DataFrame(all_cancer_data)
    column_order = [
        "Cancer Type",
        "Source URL",
        "Scrape Date",
        "Content Hash",
        "General Description",
        "Prognosis (Vooruitzichten)",
        "Symptoms (Symptomen)",
        "Causes (Oorzaken)",
        "Treatments (Behandeling)"
    ]
    df = df[column_order]
    
    df.to_csv("kanker_nl_master_table_breakout_v3.csv", index=False, encoding='utf-8-sig')
    df.to_excel("kanker_nl_master_table_breakout_v3.xlsx", index=False)
    
    print(f"\nExtraction fully updated in {time.time() - start_time:.2f} seconds!")

if __name__ == "__main__":
    main()

Fetching overview page: https://www.kanker.nl/kankersoorten

Starting parallel scraping with 5 workers...

[Processing Category] Alvleesklierkanker -> https://www.kanker.nl/kankersoorten/alvleesklierkanker

[Processing Category] Anuskanker -> https://www.kanker.nl/kankersoorten/anuskanker

[Processing Category] Atypisch fibroxanthoom -> https://www.kanker.nl/kankersoorten/atypisch-fibroxanthoom

[Processing Category] Baarmoederhalskanker -> https://www.kanker.nl/kankersoorten/baarmoederhalskanker

[Processing Category] Baarmoederkanker -> https://www.kanker.nl/kankersoorten/baarmoederkanker
--> Discovered 57 pages to scan for content.
--> Discovered 39 pages to scan for content.
--> Discovered 37 pages to scan for content.
--> Discovered 61 pages to scan for content.
--> Discovered 19 pages to scan for content.

[Processing Category] Baarmoedersarcoom -> https://www.kanker.nl/kankersoorten/baarmoedersarcoom
--> Discovered 40 pages to scan for content.

[Processing Category] Basaalcelca

In [9]:
import os
import time
from datetime import datetime
import requests
import pandas as pd
from bs4 import BeautifulSoup
import hashlib  # Added for generating the content hash

# --- REQUIRED GLOBAL PARAMETERS ---
BASE_URL = "https://www.kanker.nl"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# --- EXTENSION CONFIGURATION ---
ADDITIONAL_TARGETS = {
    "Fatigue & Consequences": "https://www.kanker.nl/gevolgen-van-kanker/vermoeidheid-bij-kanker/algemeen/vermoeidheid-bij-kanker",
    "Treatments & Radiotherapy": "https://www.kanker.nl/soorten-behandelingen/bestraling/algemeen/wat-is-bestraling-radiotherapie",
    "Advanced/Incurable Care": "https://www.kanker.nl/gevolgen-van-kanker/niet-meer-beter-worden/niet-meer-beter-worden/als-genezen-niet-meer-mogelijk-is",
    "Young Adults (16-39)": "https://www.kanker.nl/jong"
}

def discover_portal_links(root_url):
    """
    Scans a portal track page to dynamically extract all nested article 
    and sub-category documentation links to ensure nothing is missed.
    """
    discovered = {root_url}
    try:
        response = requests.get(root_url, headers=HEADERS, timeout=15)
        if response.status_code != 200:
            return list(discovered)
            
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Target main body, article structures, and primary side sub-navigation panels
        search_containers = soup.find_all(['nav', 'article', 'main', 'div'], class_=lambda x: x and any(
            c in x.lower() for c in ['menu', 'sidebar', 'content', 'links', 'grid', 'blocks']
        )) or [soup]
        
        for container in search_containers:
            for a in container.find_all('a', href=True):
                href = a['href']
                
                # Filter for relative site navigation paths cleanly
                if href.startswith('/') and not any(x in href for x in ['/gebruiker', '/zoek', '/account', '#']):
                    full_url = BASE_URL + href
                    # Ensure we stay broad within structural content frameworks
                    if any(path in full_url for path in ['/gevolgen-van-kanker', '/soorten-behandelingen', '/jong']):
                        discovered.add(full_url)
                        
    except Exception as e:
        print(f"Directory discovery warning for {root_url}: {e}")
        
    return list(discovered)

def scrape_topic_article(category_label, article_url):
    """
    Dives deeply into an article, pulling clean structured titles 
    and multi-paragraph body details text.
    """
    print(f" -> Mining Article: {article_url}")
    
    # Generate timestamp down to the second right as processing begins
    scrape_date = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    row = {
        "Global Track": category_label,
        "Page Title": "N/A",
        "Source URL": article_url,
        "Scrape Date": scrape_date,
        "Content Hash": "N/A",
        "Article Content": "N/A"
    }
    
    try:
        res = requests.get(article_url, headers=HEADERS, timeout=15)
        if res.status_code != 200:
            return None
            
        soup = BeautifulSoup(res.text, 'html.parser')
        
        # Grab clear explicit document Title
        title_tag = soup.find('h1')
        if title_tag:
            row["Page Title"] = title_tag.get_text(strip=True)
            
        # Target content blocks
        main_body = soup.find('article') or soup.find('main') or soup
        
        content_segments = []
        # Find structural layout  containing readable documentation blocks
        for element in main_body.find_all(['p', 'h2', 'h3', 'ul', 'ol']):
            # Filter  layout text boilerplate out
            txt = element.get_text(" ", strip=True)
            if not txt or any(phrase in txt.lower() for phrase in ["printen", "opslaan", "deze informatie is gecontroleerd"]):
                continue
                
            if element.name in ['h2', 'h3']:
                content_segments.append(f"\n[{txt}]\n")
            else:
                content_segments.append(txt)
                
        compiled_text = "\n".join(content_segments).strip()
        if compiled_text:
            row["Article Content"] = compiled_text
            
            # Generate SHA-256 Content Hash based on the compiled article content
            row["Content Hash"] = hashlib.sha256(compiled_text.encode('utf-8')).hexdigest()
            return row
            
    except Exception as e:
        print(f"Failed extraction execution on link {article_url}: {e}")
        
    return None

def additional_cancer_information():
    """
    Main loop logic mapping out side effects, 
    treatments, and age groups data tracks.
    """
    print("\n" + "="*50)
    print("STARTING EXTRACTION FOR ADDITIONAL CANCER INFORMATION PORTALS")
    print("="*50)
    
    extended_dataset = []
    
    for category, root_url in ADDITIONAL_TARGETS.items():
        print(f"\n[Portal Track] Analyzing structure for: {category}")
        
        # Step 1: Map all links dynamically
        links_to_crawl = discover_portal_links(root_url)
        print(f"Found {len(links_to_crawl)} nested informational paths for {category}.")
        
        # Step 2: Extract text components safely from pages found
        for target_link in links_to_crawl:
            data = scrape_topic_article(category, target_link)
            if data and data["Article Content"] != "N/A":
                extended_dataset.append(data)
            time.sleep(0.7) # Modest rate-limiting fallback delay
            
    if extended_dataset:
        df_additional = pd.DataFrame(extended_dataset)
        
        # Enforce clean sequential ordering of columns across dataframe outputs
        column_order = [
            "Global Track",
            "Page Title",
            "Source URL",
            "Scrape Date",
            "Content Hash",
            "Article Content"
        ]
        df_additional = df_additional[column_order]
        
        # Export tables to clean static file paths (Removed run_ts timestamp from names)
        csv_path = "additional_cancer_information.csv"
        xlsx_path = "additional_cancer_information.xlsx"
        
        df_additional.to_csv(csv_path, index=False, encoding='utf-8-sig')
        df_additional.to_excel(xlsx_path, index=False)
        print(f"\n[Success] Portal Track Data exported: {csv_path}, {xlsx_path}")
    else:
        print("\n[Error] No tracking records generated for the additional dataset profiles.")

# --- INJECT SYSTEM RUN TRIGGER ---
if __name__ == "__main__":
    additional_cancer_information()


STARTING EXTRACTION FOR ADDITIONAL CANCER INFORMATION PORTALS

[Portal Track] Analyzing structure for: Fatigue & Consequences
Found 12 nested informational paths for Fatigue & Consequences.
 -> Mining Article: https://www.kanker.nl/gevolgen-van-kanker/vermoeidheid-bij-kanker/algemeen/tips-bij-vermoeidheid
 -> Mining Article: https://www.kanker.nl/gevolgen-van-kanker/vermoeidheid-bij-kanker/algemeen/wat-te-doen-tegen-vermoeidheid
 -> Mining Article: https://www.kanker.nl/gevolgen-van-kanker/vermoeidheid-bij-kanker/algemeen
 -> Mining Article: https://www.kanker.nl//app-eu.readspeaker.com/cgi-bin/rsent?customerid=10829&lang=nl_nl&readid=main-article&url=https://www.kanker.nl/gevolgen-van-kanker/vermoeidheid-bij-kanker/algemeen/vermoeidheid-bij-kanker
 -> Mining Article: https://www.kanker.nl/gevolgen-van-kanker/vermoeidheid/wat-is/chronische-vermoeidheid-bij-kanker
 -> Mining Article: https://www.kanker.nl/gevolgen-van-kanker/vermoeidheid-bij-kanker/algemeen/chronische-vermoeidheid-na-k